# 📘 Week 10 복습과제 — Adam Optimizer 구현

> **Paper**: *Adam: A Method for Stochastic Optimization* (Kingma & Ba, 2015)  
> **제출 방법**: 빈칸을 채우고 셀을 모두 실행한 뒤 제출
> **예상 소요 시간**: 약 25~30분

---

### 📋 구성
| 섹션 | 내용 | 문제 유형 |
|------|------|-----------|
| 1 | 라이브러리 임포트 | — |
| 2 | 배경: Gradient Descent 계열 비교 | 서술형 |
| 3 | Adam 알고리즘 구현 (NumPy From Scratch) | 코드 빈칸 |
| 4 | Bias Correction 시각화 | 코드 빈칸 + 서술형 |
| 5 | 2D Loss Surface 궤적 비교 | 코드 빈칸 |
| 6 | PyTorch MNIST 학습 비교 | 코드 빈칸 + 서술형 |
| 7 | 하이퍼파라미터 민감도 분석 | 서술형 |

---
## 1. 라이브러리 임포트

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
from matplotlib.colors import LogNorm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print('✅ 라이브러리 로드 완료')

---
## 2. 배경: Gradient Descent 계열 비교

| 방법 | 설명 | 단점 |
|---|---|---|
| **SGD** | 전체/일부 데이터로 gradient 계산 | 진동, 느린 수렴 |
| **Momentum** | 이전 gradient 방향을 누적 | 하이퍼파라미터 추가 |
| **RMSProp** | gradient 제곱의 이동평균으로 lr 조정 | 2차 모멘트만 사용 |
| **Adam** | Momentum + RMSProp + Bias correction | ✅ 가장 효과적 |

---
### ✏️ Q1 [서술형]

Adam은 **1차 모멘트(m)** 와 **2차 모멘트(v)** 를 모두 활용합니다.  
각각이 기존의 어떤 옵티마이저 아이디어에서 비롯된 것인지 설명하고,  
두 모멘트가 파라미터 업데이트 식에서 어떤 역할을 하는지 서술하세요.

```
📝 답안 작성란:

**1차 모멘트(m)** 는 Momentum 옵티마이저의 아이디어에서 비롯되었다. Momentum이 이전 gradient들의 지수가중이동평균(EWMA)으로 업데이트 방향을 매끄럽게 만들어 진동을 줄이듯, Adam의 m은 g_t의 EWMA로서 gradient의 **1차 모멘트(평균 방향)** 를 추정한다. 즉 어느 방향으로 일관되게 내려가야 하는지를 알려주는 역할을 한다.

**2차 모멘트(v)** 는 RMSProp/AdaGrad 계열의 아이디어에서 비롯되었다. RMSProp이 gradient 제곱의 EWMA로 각 파라미터별 학습률을 적응적으로 조정하듯, Adam의 v는 g_t²의 EWMA로서 gradient의 **2차 모멘트(분산/크기)** 를 추정한다.

업데이트 식 θ_t = θ_{t-1} - α · m̂_t / (√v̂_t + ε) 에서, **분자의 m̂_t는 "어느 방향으로 얼마나 갈지"** 를 결정하고(Momentum 역할), **분모의 √v̂_t는 "각 파라미터별 스텝 크기를 얼마로 줄일지"** 를 결정한다(RMSProp 역할). gradient 크기가 큰 차원은 분모가 커져 스텝이 작아지고, 작은 차원은 분모가 작아져 상대적으로 더 큰 스텝을 받는다. 결과적으로 Adam은 **방향의 안정성(m)** 과 **스케일의 적응성(v)** 을 동시에 얻는다.
```

---
## 3. Adam 알고리즘 구현 (NumPy — From Scratch)

논문의 Algorithm 1을 NumPy로 직접 구현합니다.  
아래 수식을 참고하여 빈칸을 채우세요.

$$m_t = \beta_1 \cdot m_{t-1} + (1-\beta_1) \cdot g_t$$
$$v_t = \beta_2 \cdot v_{t-1} + (1-\beta_2) \cdot g_t^2$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
$$\theta_t = \theta_{t-1} - \alpha \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

---
### 🔲 Q2 [코드 빈칸] — AdamOptimizer 클래스

아래 클래스에서 `###답안###` 으로 표시된 부분을 채우세요.

In [ ]:
class AdamOptimizer:
    """
    Adam optimizer — NumPy 구현 (Kingma & Ba, 2015)

    Parameters
    ----------
    lr    : 학습률 α (default: 0.001)
    beta1 : 1차 모멘트 감쇠율 β₁ (default: 0.9)
    beta2 : 2차 모멘트 감쇠율 β₂ (default: 0.999)
    eps   : 수치 안정성 ε (default: 1e-8)
    """
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr    = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps   = eps
        self.m     = None   # 1차 모멘트 (Momentum)
        self.v     = None   # 2차 모멘트 (RMSProp)
        self.t     = 0      # 타임스텝

    def step(self, params, grads):
        if self.m is None:
            self.m = np.zeros_like(params)
            self.v = np.zeros_like(params)

        self.t += 1

        # ✅ Q2-① : 1차 모멘트 업데이트 (biased)
        self.m = self.beta1 * self.m + (1 - self.beta1) * grads

        # ✅ Q2-② : 2차 모멘트 업데이트 (biased)
        self.v = self.beta2 * self.v + (1 - self.beta2) * grads**2

        # Bias correction
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)

        # ✅ Q2-③ : 파라미터 업데이트 수식
        params = params - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

        return params

In [ ]:
# 동작 확인 — 1D 예시 (x² 최소화)
x = np.array([5.0])   # 초기값
optimizer = AdamOptimizer(lr=0.1)

history = [x[0]]
for _ in range(50):
    grad = 2 * x         # f(x) = x² → f'(x) = 2x
    x = optimizer.step(x, grad)
    history.append(x[0])

plt.figure(figsize=(8, 4))
plt.plot(history, 'o-', color='steelblue', markersize=4)
plt.axhline(0, color='red', linestyle='--', label='최솟값 (x=0)')
plt.title('Adam — f(x) = x² 최소화 (From Scratch)')
plt.xlabel('Iteration')
plt.ylabel('x 값')
plt.legend()
plt.tight_layout()
plt.show()
print(f'최종 x 값: {x[0]:.6f}  (정답: 0)')

---
## 4. Bias Correction 시각화

초기 타임스텝에서 모멘트가 0으로 초기화되어 있어 값이 **과소추정** 됩니다.  
Bias correction이 이를 어떻게 보정하는지 시각화합니다.

---
### 🔲 Q3 [코드 빈칸] — Bias Correction 보정값 계산

In [ ]:
T = 50
beta1, beta2 = 0.9, 0.999

# 가상의 gradient (상수 1.0으로 가정)
g = 1.0
m, v = 0.0, 0.0

m_biased, v_biased = [], []
m_corrected, v_corrected = [], []

for t in range(1, T+1):
    m = beta1 * m + (1 - beta1) * g
    v = beta2 * v + (1 - beta2) * g**2

    m_biased.append(m)
    v_biased.append(v)

    # ✅ Q3-① : bias correction 보정값 계산
    # 힌트: m_hat = m / (1 - beta1^t)
    m_corrected.append(m / (1 - beta1**t))
    v_corrected.append(v / (1 - beta2**t))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(m_biased,    label='m (biased)',    linestyle='--', color='tomato')
axes[0].plot(m_corrected, label='m_hat (corrected)', linestyle='-', color='steelblue')
axes[0].axhline(1.0, color='gray', linestyle=':', label='True mean (1.0)')
axes[0].set_title('1차 모멘트 (beta1=0.9)')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('값')
axes[0].legend()

axes[1].plot(v_biased,    label='v (biased)',    linestyle='--', color='tomato')
axes[1].plot(v_corrected, label='v_hat (corrected)', linestyle='-', color='steelblue')
axes[1].axhline(1.0, color='gray', linestyle=':', label='True mean (1.0)')
axes[1].set_title('2차 모멘트 (beta2=0.999)')
axes[1].set_xlabel('Timestep t')
axes[1].legend()

plt.suptitle('Bias Correction 효과: 초기 과소추정 → 보정', fontsize=13)
plt.tight_layout()
plt.show()

---
### ✏️ Q4 [서술형] — Bias Correction의 필요성

위 그래프에서 **biased 값** 과 **corrected 값** 의 차이를 관찰하세요.  
다음 두 가지를 서술하세요.

1. 왜 초기 타임스텝에서 biased 모멘트가 True mean보다 작게 나오는가? (초기화 값과 연결지어)
2. beta2=0.999인 2차 모멘트가 beta1=0.9인 1차 모멘트보다 bias correction 효과가 더 오래 지속되는 이유는?

```
📝 답안 작성란:

1. m, v는 모두 0으로 초기화된다. EWMA 식 m_t = β₁·m_{t-1} + (1-β₁)·g_t 를 t=1부터 풀어쓰면
   m_t = (1-β₁) · Σ_{i=1..t} β₁^(t-i) · g_i
   가 된다. gradient가 상수 1이라고 가정해 모든 g_i = 1을 대입하면 m_t = 1 - β₁^t 이 되어, **True mean=1 보다 항상 β₁^t 만큼 작은 값** 이 나온다. 즉 0으로 시작한 초기값의 가중치가 (1-β₁) 이 누적될 때까지 남아있어, 초기 타임스텝에서는 EWMA가 0 쪽으로 끌려가 **과소추정** 된다. bias correction은 이 1-β₁^t로 나눠줘 누락된 가중치를 보정한다.

2. 보정 인수 (1 - β^t)가 1에 충분히 가까워지는 데 걸리는 시간은 β가 1에 가까울수록 길다. β₁=0.9일 때 1-0.9^t는 t=20쯤이면 약 0.88로 1에 거의 도달하지만, β₂=0.999일 때 1-0.999^t는 t=50에서도 약 0.049, t=1000에서도 약 0.63에 불과하다. 즉 **β₂가 1에 훨씬 가깝기 때문에 과거 정보의 감쇠가 느리고**, 초기에 0이 차지하는 비중이 오래 남는다. 따라서 2차 모멘트의 bias가 1차 모멘트보다 훨씬 오래 지속되며, bias correction의 효과(보정 폭)도 더 오래 유의미하게 작용한다.
```

---
## 5. 2D Loss Surface에서 옵티마이저 궤적 비교

**Beale function** 을 Loss surface로 사용하여  
SGD / Momentum / RMSProp / Adam 의 경로를 비교합니다.  
전역 최솟값: **(3, 0.5)**

In [ ]:
# Loss function: Beale function
def beale(x, y):
    return ((1.5   - x + x*y   )**2 +
            (2.25  - x + x*y**2)**2 +
            (2.625 - x + x*y**3)**2)

def beale_grad(x, y):
    a = 1.5   - x + x*y;    da_dx = -1 + y;    da_dy = x
    b = 2.25  - x + x*y**2; db_dx = -1 + y**2; db_dy = 2*x*y
    c = 2.625 - x + x*y**3; dc_dx = -1 + y**3; dc_dy = 3*x*y**2
    gx = 2*(a*da_dx + b*db_dx + c*dc_dx)
    gy = 2*(a*da_dy + b*db_dy + c*dc_dy)
    return np.array([gx, gy])

---
### 🔲 Q5 [코드 빈칸] — Adam 궤적 시뮬레이션

아래 `run_optimizer` 함수에서 **Adam** 부분만 빈칸으로 되어 있습니다.  
Q2에서 구현한 수식을 참고하여 채우세요.

In [ ]:
def run_optimizer(name, init, n_iter=200, lr=0.002):
    x = np.array(init, dtype=float)
    path = [x.copy()]

    m = np.zeros(2); v = np.zeros(2); t = 0
    beta1, beta2, eps = 0.9, 0.999, 1e-8
    mu = 0.9

    for _ in range(n_iter):
        g = beale_grad(x[0], x[1])
        g = np.clip(g, -10, 10)

        if name == 'SGD':
            x = x - lr * g

        elif name == 'Momentum':
            m = mu * m + lr * g
            x = x - m

        elif name == 'RMSProp':
            v = beta2 * v + (1 - beta2) * g**2
            x = x - lr * g / (np.sqrt(v) + eps)

        elif name == 'Adam':
            t += 1
            # ✅ Q5-① : 1차 모멘트 업데이트
            m = beta1 * m + (1 - beta1) * g
            # ✅ Q5-② : 2차 모멘트 업데이트
            v = beta2 * v + (1 - beta2) * g**2
            m_hat = m / (1 - beta1**t)
            v_hat = v / (1 - beta2**t)
            # ✅ Q5-③ : 파라미터 업데이트
            x = x - lr * m_hat / (np.sqrt(v_hat) + eps)

        path.append(x.copy())
    return np.array(path)


# Loss surface 그리드
xs = np.linspace(-4.5, 4.5, 300)
ys = np.linspace(-1.5, 4.5, 300)
X, Y = np.meshgrid(xs, ys)
Z = beale(X, Y)

init_point = [-3.0, 3.5]
optimizers = ['SGD', 'Momentum', 'RMSProp', 'Adam']
colors     = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']

paths = {name: run_optimizer(name, init_point) for name in optimizers}

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for ax, name, color in zip(axes, optimizers, colors):
    ax.contourf(X, Y, Z, levels=np.logspace(0, 5, 30), cmap='YlOrRd', alpha=0.6, norm=LogNorm())
    ax.contour( X, Y, Z, levels=np.logspace(0, 5, 15), colors='gray', linewidths=0.5, norm=LogNorm())
    path = paths[name]
    ax.plot(path[:, 0], path[:, 1], '-', color=color, linewidth=1.5, alpha=0.8)
    ax.plot(path[0, 0], path[0, 1], 'o', color='black', markersize=8, label='Start')
    ax.plot(path[-1, 0], path[-1, 1], '*', color=color, markersize=12, label='End')
    ax.plot(3, 0.5, 'P', color='lime', markersize=12, label='Global Min (3, 0.5)')
    final_loss = beale(path[-1, 0], path[-1, 1])
    ax.set_title(f'{name}  |  Final Loss: {final_loss:.2f}')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=8)
    ax.set_xlim(-4.5, 4.5); ax.set_ylim(-1.5, 4.5)

plt.suptitle('Beale Function — 옵티마이저 궤적 비교 (200 iterations)', fontsize=14)
plt.tight_layout()
plt.show()

---
### ✏️ Q6 [서술형] — 궤적 비교 해석

위 4개의 궤적 그래프를 보고, 아래 질문에 답하세요.

SGD와 Adam의 궤적(경로 모양)이 다른 이유를 각 옵티마이저의 업데이트 방식과 연결지어 설명하세요.  
(단, Beale function은 비등방성(anisotropic) 곡면임을 참고하세요.)

```
📝 답안 작성란:

Beale function은 차원마다 곡률(Hessian의 고유값)이 크게 다른 **비등방성** 곡면이다. 한 방향으로는 협곡(ravine)처럼 가파르고, 다른 방향으로는 평평하다.

- **SGD** 는 x_{t+1} = x_t - α·g 로 모든 차원에 같은 학습률을 곱한다. 따라서 gradient가 큰 가파른 차원에서는 협곡의 벽을 왔다갔다 진동하며 튀고, gradient가 작은 평평한 차원에서는 거의 못 움직여서, 궤적이 협곡 안에서 **지그재그로 흔들리며 느리게 진행** 한다. 200 iteration 안에 전역 최솟값 (3, 0.5)까지 도달하기 어렵다.

- **Adam** 은 우선 1차 모멘트 m으로 과거 gradient를 평균내므로 협곡 벽에서 발생하는 좌우 진동 성분이 상쇄되어 **방향이 매끄러워진다**(Momentum 효과). 동시에 2차 모멘트 √v로 차원별 스텝 크기를 정규화하므로 가파른 차원에서는 분모가 커져 스텝이 작아지고, 평평한 차원에서는 분모가 작아져 스텝이 상대적으로 커진다(RMSProp 효과). 결과적으로 Adam의 업데이트는 **각 차원의 곡률에 적응** 하여 협곡을 따라 부드럽게 이동하고, 진동 없이 전역 최솟값을 향해 더 직선에 가까운 궤적으로 수렴한다.
```

---
## 6. PyTorch로 MNIST 학습 — 옵티마이저별 수렴 속도 비교

같은 네트워크 구조에서 SGD / Momentum / RMSProp / Adam 의 수렴 속도를 비교합니다.

In [ ]:
# 데이터 준비
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

print(f'Train: {len(train_dataset):,}개 | Test: {len(test_dataset):,}개')

---
### 🔲 Q7 [코드 빈칸] — Adam 옵티마이저 설정 (논문 기본값)

아래 `train_model` 함수에서 Adam 옵티마이저 생성 부분을 논문 권장 기본값으로 채우세요.  
논문 권장값: **lr=0.001, β₁=0.9, β₂=0.999, ε=1e-8**

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def train_model(optimizer_name, n_epochs=10):
    model = MLP().to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=0.01)
    elif optimizer_name == 'Momentum':
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    elif optimizer_name == 'RMSProp':
        optimizer = optim.RMSprop(model.parameters(), lr=0.001)
    elif optimizer_name == 'Adam':
        # ✅ Q7 : 논문 권장값으로 Adam 옵티마이저를 설정하세요
        optimizer = optim.Adam(model.parameters(),
                               lr=0.001,
                               betas=(0.9, 0.999),
                               eps=1e-8)

    train_losses, test_accs = [], []

    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss = 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        train_losses.append(avg_loss)

        model.eval()
        correct = 0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                pred = model(X).argmax(dim=1)
                correct += (pred == y).sum().item()
        acc = correct / len(test_dataset) * 100
        test_accs.append(acc)

        print(f'[{optimizer_name}] Epoch {epoch:2d} | Loss: {avg_loss:.4f} | Test Acc: {acc:.2f}%')

    return train_losses, test_accs

In [ ]:
# 4가지 옵티마이저 학습 실행
N_EPOCHS = 10
optimizer_names = ['SGD', 'Momentum', 'RMSProp', 'Adam']
colors_pt = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']

results = {}
for name in optimizer_names:
    print(f'\n' + '='*50)
    torch.manual_seed(SEED)
    results[name] = train_model(name, N_EPOCHS)

In [ ]:
# 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs = range(1, N_EPOCHS + 1)

for name, color in zip(optimizer_names, colors_pt):
    losses, accs = results[name]
    axes[0].plot(epochs, losses, '-o', label=name, color=color, markersize=5)
    axes[1].plot(epochs, accs,   '-o', label=name, color=color, markersize=5)

axes[0].set_title('Train Loss per Epoch')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].set_title('Test Accuracy per Epoch (%)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

plt.suptitle('MNIST — 옵티마이저별 수렴 속도 비교', fontsize=14)
plt.tight_layout()
plt.show()

print('\n📊 최종 성능 요약 (Epoch 10)')
print(f'{"Optimizer":<12} {"Train Loss":>12} {"Test Acc":>10}')
print('-' * 36)
for name in optimizer_names:
    losses, accs = results[name]
    print(f'{name:<12} {losses[-1]:>12.4f} {accs[-1]:>9.2f}%')

---
### ✏️ Q8 [서술형] — MNIST 결과 해석

위 학습 결과를 보고 다음을 서술하세요.

RMSProp의 Train Loss가 Adam보다 낮음에도 Test Accuracy는 Adam보다 낮은 경향이 나타납니다.  
이 현상이 나타날 수 있는 이유를 **generalization** 관점에서 설명하고,  
Adam이 RMSProp보다 일반화 성능이 좋을 수 있는 이유를 한 가지 이상 서술하세요.

```
📝 답안 작성란:

이는 **train loss(최적화 성능)** 와 **test accuracy(일반화 성능)** 가 항상 같이 가지는 않는 전형적인 현상이다.

RMSProp은 2차 모멘트 v만 사용해 학습률을 적응시킨다. 그래서 train set의 손실을 빠르게 떨어뜨릴 수는 있지만, gradient의 분산이 큰 noise에 더 민감하게 반응해서 **train set의 미세한 패턴까지 과도하게 fitting** 하는 경향이 있다. 즉, train loss는 더 낮아져도 test set에서는 일반화 성능이 떨어질 수 있다.

Adam이 RMSProp보다 일반화 성능이 좋을 수 있는 이유는 다음과 같다.

1. **1차 모멘트 m이 gradient를 평균내어 noise를 평활화** 한다. 미니배치 gradient의 확률적 잡음이 m을 통해 상쇄되므로, RMSProp처럼 매 스텝의 noise를 그대로 반영하지 않는다. 이는 일종의 **암묵적 정규화(implicit regularization)** 로 작용해 sharp minimum을 피하고 더 평평한(flat) minimum 쪽으로 수렴하게 한다. flat minimum은 일반적으로 일반화 성능이 더 좋다고 알려져 있다.

2. **Bias correction을 통해 초기 학습 단계에서 과도한 스텝을 막아** 안정적으로 수렴한다. RMSProp은 bias correction이 없어 초기에 v가 과소추정되면 분모가 작아져 스텝이 비정상적으로 커질 수 있고, 이런 초기 큰 스텝이 sharp minimum으로 빠져들 위험을 만든다.

3. RMSProp은 매 스텝 gradient g를 그대로 분자에 쓰므로 진동(oscillation)이 train loss에는 도움이 될 수도 있지만 일반화에는 손해다. Adam은 m̂으로 진동을 줄여 **더 균형 잡힌 업데이트** 를 한다.
```

---
## 7. 하이퍼파라미터 민감도 분석

논문 권장 default: **α=0.001, β₁=0.9, β₂=0.999, ε=1e-8**

---
### 🔲 Q9 [코드 빈칸] — 학습률(lr) 민감도 실험

In [ ]:
def train_adam_with_lr(lr, n_epochs=10):
    torch.manual_seed(SEED)
    model = MLP().to(device)
    criterion = nn.CrossEntropyLoss()
    # ✅ Q9 : lr 인자를 사용해 Adam 옵티마이저를 생성하세요
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []
    for _ in range(n_epochs):
        model.train()
        total = 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            total += loss.item()
        losses.append(total / len(train_loader))
    return losses

lr_values = [0.1, 0.01, 0.001, 0.0001]
lr_colors = ['#e74c3c', '#f39c12', '#3498db', '#9b59b6']

print('학습률 실험 중...')
lr_results = {}
for lr in lr_values:
    print(f'  lr={lr}')
    lr_results[lr] = train_adam_with_lr(lr)

plt.figure(figsize=(9, 5))
for lr, color in zip(lr_values, lr_colors):
    label = f'lr={lr}' + (' ← 논문 권장' if lr == 0.001 else '')
    plt.plot(range(1, 11), lr_results[lr], '-o', label=label, color=color, markersize=5)

plt.title('Adam — 학습률(α)에 따른 Train Loss 변화')
plt.xlabel('Epoch'); plt.ylabel('Train Loss')
plt.legend(); plt.tight_layout()
plt.show()

In [ ]:
# β₁ 변화에 따른 영향
def train_adam_with_beta1(beta1, n_epochs=10):
    torch.manual_seed(SEED)
    model = MLP().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(beta1, 0.999))
    losses = []
    for _ in range(n_epochs):
        model.train(); total = 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward(); optimizer.step()
            total += loss.item()
        losses.append(total / len(train_loader))
    return losses

beta1_values = [0.5, 0.7, 0.9, 0.95]
beta1_colors = ['#e74c3c', '#f39c12', '#3498db', '#9b59b6']

print('β₁ 실험 중...')
beta1_results = {}
for b1 in beta1_values:
    print(f'  β₁={b1}')
    beta1_results[b1] = train_adam_with_beta1(b1)

plt.figure(figsize=(9, 5))
for b1, color in zip(beta1_values, beta1_colors):
    label = f'β₁={b1}' + (' ← 논문 권장' if b1 == 0.9 else '')
    plt.plot(range(1, 11), beta1_results[b1], '-o', label=label, color=color, markersize=5)

plt.title('Adam — β₁ (1차 모멘트 감쇠율)에 따른 Train Loss')
plt.xlabel('Epoch'); plt.ylabel('Train Loss')
plt.legend(); plt.tight_layout()
plt.show()

print('\n💡 논문 권장 하이퍼파라미터:')
print('   α  = 0.001')
print('   β₁ = 0.9')
print('   β₂ = 0.999')
print('   ε  = 1e-8')

---
### ✏️ Q10 [서술형] — 하이퍼파라미터 분석

위 두 실험(lr 민감도, β₁ 민감도) 그래프를 보고 아래 질문에 답하세요.

1. **lr=0.1** 일 때 loss가 수렴하지 않고 높게 유지되는 이유를 수식과 연결지어 설명하세요.
2. **β₁=0.5** 일 때 β₁=0.9에 비해 학습이 덜 안정적인 이유는 무엇인가요?

```
📝 답안 작성란:

1. Adam의 업데이트 식 θ_t = θ_{t-1} - α · m̂_t / (√v̂_t + ε) 에서 **학습률 α는 전체 스텝 크기를 곱하는 스케일** 이다. 분모 √v̂는 gradient 크기에 따라 차원별 스텝을 정규화해 주지만, 정규화된 스텝의 절대 크기 자체는 결국 α로 정해진다. α=0.1은 논문 권장값 0.001의 100배로, 매 스텝마다 손실 함수의 곡률 스케일을 넘는 큰 점프가 발생한다. 그 결과 minimum 근처에서 stochastic noise까지 증폭되어 손실 표면의 골짜기를 넘나들며 발산하거나 진동만 하고 수렴하지 못한다. 또한 학습 초기에는 bias correction에 의해 m̂, v̂가 한 번 더 커지므로 큰 α의 영향이 더욱 증폭되어 첫 epoch부터 loss가 폭주하기 쉽다.

2. β₁은 1차 모멘트의 EWMA 감쇠율로, **과거 gradient를 얼마나 길게 평균낼지** 를 결정한다. EWMA의 유효 평균 길이는 대략 1/(1-β₁)이다. β₁=0.9이면 약 10 스텝의 gradient를 평균내지만, β₁=0.5이면 약 2 스텝만 평균낸다. 즉 β₁=0.5는 거의 매 스텝의 gradient를 그대로 따라가므로, 미니배치마다 발생하는 stochastic noise가 m̂에 거의 그대로 반영된다. 이로 인해 업데이트 방향의 변동이 커지고 진동이 심해져 학습이 덜 안정적으로 보인다. 반면 β₁=0.9는 noise를 충분히 평활화해 매끄러운 업데이트 방향을 제공하므로 학습이 더 안정적이다.
```

---
## 📋 제출 전 체크리스트

- [ ] Q2 `AdamOptimizer.step()` — 모멘트 업데이트 & 파라미터 업데이트 빈칸 완성
- [ ] Q3 Bias Correction 보정값 계산 빈칸 완성
- [ ] Q5 Adam 궤적 시뮬레이션 빈칸 완성
- [ ] Q7 PyTorch Adam 하이퍼파라미터 빈칸 완성
- [ ] Q9 학습률 민감도 실험 빈칸 완성
- [ ] Q1, Q4, Q6, Q8, Q10 서술형 답안 작성
- [ ] 모든 셀 실행 완료 (Run All)

---
**참고 자료**: Kingma, D. P., & Ba, J. (2015). *Adam: A Method for Stochastic Optimization*. ICLR 2015.